<a href="https://colab.research.google.com/github/saiteja-1919/RAG-VS-NON-RAG-MODEL/blob/main/NLP_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pdfplumber sentence-transformers faiss-cpu transformers torch torchvision torchaudio accelerate reportlab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.3 MB/s eta 0:00:00


In [ ]:
pip install pypdf sentence-transformers faiss-cpu numpy transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 7.5 MB/s eta 0:00:00


In [13]:
# ================================================================
#   RAG vs Non-RAG Comparison
#   PDF-based Question Answering using FAISS + Flan-T5
# ================================================================

# ---------- Imports ----------
import numpy as np
import faiss
import re
from collections import Counter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ================================================================
#  SECTION 1 — PDF INGESTION
# ================================================================

PDF_PATH = "/content/MACHINE LEARNING(R17A0534) (1).pdf"

def read_pdf(filepath):
    """Extract all text from a PDF file."""
    reader = PdfReader(filepath)
    full_text = ""
    for pg in reader.pages:
        extracted = pg.extract_text()
        if extracted:
            full_text += extracted + "\n"
    return full_text

print(">>> Reading PDF...")
raw_text = read_pdf(PDF_PATH)
print(f">>> Done! Total characters extracted: {len(raw_text)}\n")


# ================================================================
#  SECTION 2 — TEXT CHUNKING
# ================================================================

def split_into_chunks(text, chunk_size=220, overlap=40):
    """
    Split text into overlapping word-based chunks.
    Overlap helps preserve context at chunk boundaries.
    """
    words = text.split()
    step = chunk_size - overlap
    segments = []
    for start in range(0, len(words), step):
        segment = " ".join(words[start : start + chunk_size])
        segments.append(segment)
    return segments

text_chunks = split_into_chunks(raw_text)
print(f">>> Total chunks created: {len(text_chunks)}\n")


# ================================================================
#  SECTION 3 — EMBEDDING GENERATION
# ================================================================

print(">>> Generating embeddings...")
encoder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = encoder.encode(text_chunks, show_progress_bar=True)
print(">>> Embeddings ready!\n")


# ================================================================
#  SECTION 4 — VECTOR INDEX (FAISS)
# ================================================================

vec_dim = chunk_vectors.shape[1]
vector_store = faiss.IndexFlatL2(vec_dim)
vector_store.add(np.array(chunk_vectors))
print(f">>> FAISS index built with {vector_store.ntotal} vectors\n")


# ================================================================
#  SECTION 5 — LANGUAGE MODEL SETUP
# ================================================================

MODEL_NAME = "google/flan-t5-base"

print(f">>> Loading language model: {MODEL_NAME}...")
lm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
lm_model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(">>> Language model loaded!\n")

def run_model(prompt_text, max_tokens=200):
    """Tokenize the prompt and return generated answer."""
    tokens = lm_tokenizer(prompt_text, return_tensors="pt", truncation=True)
    output = lm_model.generate(**tokens, max_new_tokens=max_tokens)
    return lm_tokenizer.decode(output[0], skip_special_tokens=True)


# ================================================================
#  SECTION 6 — RAG PIPELINE
# ================================================================

def rag_answer(query, top_k=3, context_limit=1200):
    """
    Retrieval-Augmented Generation:
    1. Embed the query
    2. Retrieve top-k relevant chunks from FAISS
    3. Build a grounded prompt and generate an answer
    """
    query_vec = encoder.encode([query])
    _, matched_ids = vector_store.search(np.array(query_vec), k=top_k)

    retrieved_context = " ".join(
        [text_chunks[idx] for idx in matched_ids[0]]
    )[:context_limit]

    prompt = (
        "You are a helpful AI tutor explaining concepts from a machine learning textbook.\n"
        "Use the provided context to give a clear, complete answer in 3 to 5 sentences.\n"
        "Use simple language.\n\n"
        f"Context:\n{retrieved_context}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return run_model(prompt).strip()


# ================================================================
#  SECTION 7 — NON-RAG PIPELINE
# ================================================================

def plain_answer(query):
    """
    Non-RAG baseline:
    Directly feeds the question to the model with no retrieved context.
    """
    prompt = (
        "You are a helpful AI tutor with knowledge of machine learning.\n"
        "Answer the following question in 3 to 5 sentences using simple language.\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    return run_model(prompt).strip()


# ================================================================
#  SECTION 8 — EVALUATION METRICS
# ================================================================

def tokenize(text):
    """Lowercase and split text into word tokens."""
    return re.findall(r'\b\w+\b', text.lower())

def rouge_l_score(reference, hypothesis):
    """
    Compute ROUGE-L F1 score based on Longest Common Subsequence (LCS).
    Measures how much the answer overlaps with a reference answer.
    """
    ref_tokens  = tokenize(reference)
    hyp_tokens  = tokenize(hypothesis)
    r, h        = len(ref_tokens), len(hyp_tokens)

    # Build LCS table
    dp = [[0] * (h + 1) for _ in range(r + 1)]
    for i in range(1, r + 1):
        for j in range(1, h + 1):
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[r][h]

    precision = lcs / h if h else 0
    recall    = lcs / r if r else 0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0
    return round(f1, 4)

def bleu_score(reference, hypothesis, max_n=2):
    """
    Compute a simplified BLEU score up to max_n n-grams.
    Measures how many n-grams in the hypothesis appear in the reference.
    """
    ref_tokens = tokenize(reference)
    hyp_tokens = tokenize(hypothesis)

    if not hyp_tokens:
        return 0.0

    scores = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(
            tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)
        )
        hyp_ngrams = Counter(
            tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1)
        )
        matches = sum((hyp_ngrams & ref_ngrams).values())
        total   = sum(hyp_ngrams.values())
        scores.append(matches / total if total else 0)

    bleu = (scores[0] * scores[1]) ** 0.5 if len(scores) == 2 else scores[0]
    return round(bleu, 4)

def semantic_similarity(text_a, text_b):
    """
    Cosine similarity between two sentence embeddings.
    Score ranges from 0 (no similarity) to 1 (identical meaning).
    """
    vecs = encoder.encode([text_a, text_b])
    cos  = np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1]))
    return round(float(cos), 4)

def answer_length_ratio(answer):
    """Word count as a proxy for answer completeness."""
    return len(tokenize(answer))

def print_evaluation(question, rag_ans, plain_ans, reference=None):
    """
    Print a formatted evaluation table comparing RAG vs Non-RAG.
    If a reference answer is provided, ROUGE-L and BLEU are computed against it.
    Semantic similarity is always computed between the two answers.
    """
    sem_sim      = semantic_similarity(rag_ans, plain_ans)
    rag_length   = answer_length_ratio(rag_ans)
    plain_length = answer_length_ratio(plain_ans)

    print("\n" + "=" * 60)
    print("          EVALUATION METRICS REPORT")
    print("=" * 60)

    if reference:
        rag_rouge   = rouge_l_score(reference, rag_ans)
        plain_rouge = rouge_l_score(reference, plain_ans)
        rag_bleu    = bleu_score(reference, rag_ans)
        plain_bleu  = bleu_score(reference, plain_ans)

        print(f"\n{'Metric':<28} {'RAG':>10} {'Non-RAG':>10}  {'Winner':>8}")
        print("-" * 60)
        print(f"{'ROUGE-L (vs reference)':<28} {rag_rouge:>10} {plain_rouge:>10}  "
              f"{'✅ RAG' if rag_rouge >= plain_rouge else '❌ Non-RAG':>8}")
        print(f"{'BLEU (vs reference)':<28} {rag_bleu:>10} {plain_bleu:>10}  "
              f"{'✅ RAG' if rag_bleu >= plain_bleu else '❌ Non-RAG':>8}")
    else:
        print("\n  (No reference answer provided — skipping ROUGE-L & BLEU)")
        print(f"\n{'Metric':<28} {'RAG':>10} {'Non-RAG':>10}  {'Winner':>8}")
        print("-" * 60)

    print(f"{'Semantic Similarity':<28} {'(between RAG and Non-RAG answers)':>30}")
    print(f"  Score: {sem_sim}  →  "
          f"{'High agreement ✅' if sem_sim > 0.8 else 'Moderate agreement ⚠️' if sem_sim > 0.5 else 'Low agreement ❌'}")

    print(f"\n{'Answer Word Count':<28} {rag_length:>10} {plain_length:>10}  "
          f"{'✅ RAG' if rag_length >= plain_length else '❌ Non-RAG':>8}")

    # Overall verdict
    if reference:
        rag_score   = sum([rag_rouge >= plain_rouge, rag_bleu >= plain_bleu, rag_length >= plain_length])
        plain_score = 3 - rag_score
        print(f"\n{'OVERALL SCORE':<28} {rag_score:>10} {plain_score:>10}")
        winner = "✅ RAG wins!" if rag_score > plain_score else "❌ Non-RAG wins!" if plain_score > rag_score else "🤝 Tie!"
        print(f"  → {winner}")

    print("=" * 60 + "\n")


# ================================================================
#  SECTION 9 — INTERACTIVE COMPARISON LOOP
# ================================================================

print("=" * 60)
print("   RAG vs Non-RAG — Interactive Q&A + Evaluation")
print("   Type 'exit' to quit")
print("=" * 60 + "\n")

# Storage for final summary
session_log = []   # each entry: dict with question, rag, plain, metrics

while True:
    user_query = input("Your Question: ").strip()
    if user_query.lower() == "exit":
        break
    if not user_query:
        continue

    print("\n[WITHOUT RAG]")
    non_rag_result = plain_answer(user_query)
    print(non_rag_result)

    print("\n[WITH RAG]")
    rag_result = rag_answer(user_query)
    print(rag_result)

    # Optional reference answer
    ref = input("\nProvide a reference answer for full metrics (or press Enter to skip): ").strip()
    reference_answer = ref if ref else None

    print_evaluation(user_query, rag_result, non_rag_result, reference=reference_answer)

    # --- Collect metrics for final summary ---
    sem  = semantic_similarity(rag_result, non_rag_result)
    rl_r = rouge_l_score(reference_answer, rag_result)    if reference_answer else None
    rl_p = rouge_l_score(reference_answer, non_rag_result) if reference_answer else None
    bl_r = bleu_score(reference_answer, rag_result)       if reference_answer else None
    bl_p = bleu_score(reference_answer, non_rag_result)   if reference_answer else None
    wc_r = answer_length_ratio(rag_result)
    wc_p = answer_length_ratio(non_rag_result)

    session_log.append({
        "question"  : user_query,
        "rag_ans"   : rag_result,
        "plain_ans" : non_rag_result,
        "sem_sim"   : sem,
        "rouge_rag" : rl_r,  "rouge_plain": rl_p,
        "bleu_rag"  : bl_r,  "bleu_plain" : bl_p,
        "wc_rag"    : wc_r,  "wc_plain"   : wc_p,
    })


# ================================================================
#  SECTION 10 — FINAL EVALUATION MATRIX (shown on exit)
# ================================================================

if not session_log:
    print("\nNo questions were asked. Goodbye!")
else:
    print("\n\n")
    print("╔" + "═" * 70 + "╗")
    print("║" + "   FINAL EVALUATION MATRIX — ALL QUESTIONS SUMMARY".center(70) + "║")
    print("╚" + "═" * 70 + "╝")

    total_q       = len(session_log)
    rag_rouge_wins  = 0;  plain_rouge_wins  = 0;  rouge_count  = 0
    rag_bleu_wins   = 0;  plain_bleu_wins   = 0;  bleu_count   = 0
    rag_wc_wins     = 0;  plain_wc_wins     = 0
    sem_scores      = []
    avg_rag_rouge   = [];  avg_plain_rouge  = []
    avg_rag_bleu    = [];  avg_plain_bleu   = []
    avg_rag_wc      = [];  avg_plain_wc     = []

    print(f"\n  Total Questions Asked: {total_q}\n")
    print(f"  {'#':<4} {'Question (truncated)':<32} {'Sem.Sim':>8} {'ROUGE RAG':>10} {'ROUGE Non':>10} {'BLEU RAG':>9} {'BLEU Non':>9} {'WC RAG':>7} {'WC Non':>7}")
    print("  " + "-" * 98)

    for i, entry in enumerate(session_log, 1):
        q_short  = entry["question"][:30] + ".." if len(entry["question"]) > 30 else entry["question"]
        rouge_r  = f"{entry['rouge_rag']:.4f}"  if entry["rouge_rag"]  is not None else "  N/A "
        rouge_p  = f"{entry['rouge_plain']:.4f}" if entry["rouge_plain"] is not None else "  N/A "
        bleu_r   = f"{entry['bleu_rag']:.4f}"   if entry["bleu_rag"]   is not None else "  N/A "
        bleu_p   = f"{entry['bleu_plain']:.4f}"  if entry["bleu_plain"]  is not None else "  N/A "

        print(f"  {i:<4} {q_short:<32} {entry['sem_sim']:>8.4f} {rouge_r:>10} {rouge_p:>10} {bleu_r:>9} {bleu_p:>9} {entry['wc_rag']:>7} {entry['wc_plain']:>7}")

        sem_scores.append(entry["sem_sim"])
        avg_rag_wc.append(entry["wc_rag"])
        avg_plain_wc.append(entry["wc_plain"])
        if entry["wc_rag"] >= entry["wc_plain"]: rag_wc_wins += 1
        else: plain_wc_wins += 1

        if entry["rouge_rag"] is not None:
            avg_rag_rouge.append(entry["rouge_rag"])
            avg_plain_rouge.append(entry["rouge_plain"])
            rouge_count += 1
            if entry["rouge_rag"] >= entry["rouge_plain"]: rag_rouge_wins += 1
            else: plain_rouge_wins += 1

        if entry["bleu_rag"] is not None:
            avg_rag_bleu.append(entry["bleu_rag"])
            avg_plain_bleu.append(entry["bleu_plain"])
            bleu_count += 1
            if entry["bleu_rag"] >= entry["bleu_plain"]: rag_bleu_wins += 1
            else: plain_bleu_wins += 1

    # --- Averages Row ---
    print("  " + "-" * 98)
    avg_sem    = sum(sem_scores) / len(sem_scores)
    avg_rr     = sum(avg_rag_rouge)   / len(avg_rag_rouge)   if avg_rag_rouge   else None
    avg_rp     = sum(avg_plain_rouge) / len(avg_plain_rouge) if avg_plain_rouge else None
    avg_br     = sum(avg_rag_bleu)    / len(avg_rag_bleu)    if avg_rag_bleu    else None
    avg_bp     = sum(avg_plain_bleu)  / len(avg_plain_bleu)  if avg_plain_bleu  else None
    avg_wr     = sum(avg_rag_wc)      / len(avg_rag_wc)
    avg_wp     = sum(avg_plain_wc)    / len(avg_plain_wc)

    rr_str = f"{avg_rr:.4f}" if avg_rr is not None else "  N/A "
    rp_str = f"{avg_rp:.4f}" if avg_rp is not None else "  N/A "
    br_str = f"{avg_br:.4f}" if avg_br is not None else "  N/A "
    bp_str = f"{avg_bp:.4f}" if avg_bp is not None else "  N/A "

    print(f"  {'AVG':<4} {'(across all questions)':<32} {avg_sem:>8.4f} {rr_str:>10} {rp_str:>10} {br_str:>9} {bp_str:>9} {avg_wr:>7.1f} {avg_wp:>7.1f}")

    # --- Win Count Summary ---
    print("\n")
    print("  ┌─────────────────────────────────────────────────────┐")
    print("  │              WIN COUNT SUMMARY                      │")
    print("  ├──────────────────────┬──────────────┬──────────────┤")
    print("  │ Metric               │     RAG      │   Non-RAG    │")
    print("  ├──────────────────────┼──────────────┼──────────────┤")

    def win_cell(r, p, total):
        return f"{r}/{total} {'✅' if r > p else '  '}", f"{p}/{total} {'✅' if p > r else '  '}"

    if rouge_count:
        rc, pc = win_cell(rag_rouge_wins, plain_rouge_wins, rouge_count)
        print(f"  │ {'ROUGE-L':<20} │ {rc:^12} │ {pc:^12} │")

    if bleu_count:
        rc, pc = win_cell(rag_bleu_wins, plain_bleu_wins, bleu_count)
        print(f"  │ {'BLEU':<20} │ {rc:^12} │ {pc:^12} │")

    rc, pc = win_cell(rag_wc_wins, plain_wc_wins, total_q)
    print(f"  │ {'Word Count':<20} │ {rc:^12} │ {pc:^12} │")
    print("  ├──────────────────────┼──────────────┼──────────────┤")

    # Tally total wins
    total_rag_wins   = rag_rouge_wins + rag_bleu_wins + rag_wc_wins
    total_plain_wins = plain_rouge_wins + plain_bleu_wins + plain_wc_wins
    total_possible   = rouge_count + bleu_count + total_q
    rc, pc = win_cell(total_rag_wins, total_plain_wins, total_possible)
    print(f"  │ {'TOTAL WINS':<20} │ {rc:^12} │ {pc:^12} │")
    print("  └──────────────────────┴──────────────┴──────────────┘")

    # Final verdict
    print("\n  ── SEMANTIC SIMILARITY (avg across all questions) ──")
    print(f"  Score: {avg_sem:.4f}  →  "
          f"{'High agreement ✅' if avg_sem > 0.8 else 'Moderate agreement ⚠️' if avg_sem > 0.5 else 'Low agreement ❌'}")

    print("\n  ── OVERALL VERDICT ──")
    if total_rag_wins > total_plain_wins:
        print("  🏆  RAG outperformed Non-RAG across this session!")
    elif total_plain_wins > total_rag_wins:
        print("  ⚠️   Non-RAG outperformed RAG — consider tuning retrieval.")
    else:
        print("  🤝  Both approaches performed equally in this session.")

    print("\n" + "═" * 72)
    print("  Session complete. Goodbye!")
    print("═" * 72 + "\n")

>>> Reading PDF...
>>> Done! Total characters extracted: 243843

>>> Total chunks created: 221

>>> Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

>>> Embeddings ready!

>>> FAISS index built with 221 vectors

>>> Loading language model: google/flan-t5-base...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


>>> Language model loaded!

   RAG vs Non-RAG — Interactive Q&A + Evaluation
   Type 'exit' to quit

Your Question: what is svm

[WITHOUT RAG]
a machine learning algorithm

[WITH RAG]
Supervised Learning algorithms

Provide a reference answer for full metrics (or press Enter to skip): 

          EVALUATION METRICS REPORT

  (No reference answer provided — skipping ROUGE-L & BLEU)

Metric                              RAG    Non-RAG    Winner
------------------------------------------------------------
Semantic Similarity          (between RAG and Non-RAG answers)
  Score: 0.7487  →  Moderate agreement ⚠️

Answer Word Count                     3          4  ❌ Non-RAG

Your Question: What is overfitting? 

[WITHOUT RAG]
a t-shirt

[WITH RAG]
If our algorithm works well with the training dataset but not well with test dataset, then such problem is called Overfitting.

Provide a reference answer for full metrics (or press Enter to skip): 

          EVALUATION METRICS REPORT

  (No referen